# Lecture 8 · Notebook 1: Pipelines and the Hub

Natural Language Processing (2026), SeoulTech.

A `pipeline` hides three steps: **tokenizer → model → post-processing**.
We first use pipelines for several tasks, then rebuild one pipeline by hand.

*Runtime: CPU is enough. The first call of each model downloads its files from the Hub (a few hundred MB in total).*

In [2]:
# Run this cell first. Colab already has PyTorch; this installs the tested library versions.
%pip install -q "transformers>=5.20,<6" "datasets>=5.1,<6" "accelerate>=1.10"

import torch, transformers, datasets
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
transformers 5.21.1 | datasets 5.1.0 | torch 2.15.0
device: mps 


## 1. Sentiment analysis

`pipeline(task, model=...)` downloads the model and its tokenizer from the Hub and caches them
(in `~/.cache/huggingface`). We always name the model explicitly, so the result does not change
when the library's default model changes.

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis",
               model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
clf("I was not expecting much, but this movie was wonderful.")

#clf의 의미: DistiBERT모델을 가지고 옴 clf 는 classfier의 줄임 말로 붙인 변수 이름.  clf 라는 객체에 (문장)을 넣음
#결과는 label 과 score 값

W1029 21:41:18.608000 4042 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 10211.09it/s]


[{'label': 'POSITIVE', 'score': 0.9998770952224731}]

A pipeline also takes a list of inputs and returns one result per input.

In [ ]:
reviews = [
    "The plot was predictable and the acting was flat.",
    "A quiet film that stays with you long after it ends.",
    "Not bad.",
]
for r, out in zip(reviews, clf(reviews)):
    print(f"{out['label']:>8}  {out['score']:.3f}  {r}")

# reviews에 list로 문장이 들어가 있고 , clf 를 통해서 label과 score 가 있음

NEGATIVE  1.000  The plot was predictable and the acting was flat.
POSITIVE  0.999  A quiet film that stays with you long after it ends.
POSITIVE  0.999  Not bad.


**Look at "Not bad."** The model was fine-tuned on SST-2 (movie-review sentences).
Short or unusual inputs can get confident but questionable labels: the score is a softmax output,
not a guarantee.

## 2. Other tasks

| Task name | What it returns | Model used here |
|---|---|---|
| `token-classification` (`ner`) | a label per token or word span | BERT-large fine-tuned on CoNLL-2003 |
| `fill-mask` | the most likely words for `[MASK]` | BERT base (uncased) |
| `zero-shot-classification` | scores for labels you choose at run time | BART-large fine-tuned on MNLI |
| `text-generation` | a continuation, or a chat reply | SmolLM2-360M-Instruct |

In [ ]:
ner = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english",
               aggregation_strategy="simple")   # merge word pieces into whole entities
ner("Hugging Face was founded in New York by Clément Delangue and Julien Chaumond.")

#ner = named entity recognition => 개채명 인식
#ex) hugging face 조직, new york 장소 clement delangue 사람, julien chaumond ㅏ사람
#start와 end는 문장안에서 해당 개체명이 차지하는 문자위치
#aggregation_startegy ="simple" 토큰 조각들을 하나의 개채명으로 합치는 방식 -> 연속도니 토큰 결과를 간단한 규칙으로 합쳐서 사람이 읽기 쉬운 단어 또는 구단위로 변환해라

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6895.71it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'ORG',
  'score': np.float32(0.9589004),
  'word': 'Hugging Face',
  'start': 0,
  'end': 12},
 {'entity_group': 'LOC',
  'score': np.float32(0.99894726),
  'word': 'New York',
  'start': 28,
  'end': 36},
 {'entity_group': 'PER',
  'score': np.float32(0.99829143),
  'word': 'Clément Delangue',
  'start': 40,
  'end': 56},
 {'entity_group': 'PER',
  'score': np.float32(0.9955491),
  'word': 'Julien Chaumond',
  'start': 61,
  'end': 76}]

In [7]:
fill = pipeline("fill-mask", model="google-bert/bert-base-uncased")
for out in fill("Seoul is the capital of [MASK] Korea."):
    print(f"{out['token_str']:>12}  {out['score']:.3f}")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 9865.73it/s]
[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


       south  0.997
       north  0.002
       seoul  0.000
     central  0.000
    southern  0.000


Zero-shot classification uses a model trained for natural language inference (Lecture 7, slide 13):
each candidate label is turned into a hypothesis such as *"This example is about sports."*,
and the entailment probability becomes the label's score.

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
zs("The new phone has a brighter screen and a battery that lasts two days.",
   candidate_labels=["technology", "sports", "politics", "cooking"])

#내부적으로 각후보를 이런 문장으로 바꿔서 생각한다고 함
"""This example is about technology.
This example is about sports.
This example is about politics.
This example is about cooking."""

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 11011.31it/s]


{'sequence': 'The new phone has a brighter screen and a battery that lasts two days.',
 'labels': ['technology', 'sports', 'politics', 'cooking'],
 'scores': [0.9782903790473938,
  0.013325541280210018,
  0.005072557367384434,
  0.0033114596735686064]}

A text-generation pipeline with an instruction-tuned model accepts a list of chat messages
(Lecture 7). The pipeline applies the model's chat template for us; Notebook 3 shows what that means.

In [ ]:
chat = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device=device)
messages = [{"role": "user", "content": "Explain the moon landing to a 6 year old in a few sentences."}]
out = chat(messages, max_new_tokens=80, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

#이 instruct 모델이라서 단순히 다음 단어를 잇는것 보다 사용자의 지시를 읽고 답변을 생성하도록 추가 학습이 되어있음 

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10498.88it/s]


The moon landing was a really cool event where a group of people called astronauts went to the moon and did some amazing things. They landed on the moon and explored it for a long time. It was a big deal because it showed that humans can do really cool things in space.


## 3. Inside a pipeline

We now reproduce the sentiment pipeline by hand: tokenizer → model → softmax → label name.

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
#automodelforsequenceclassification => 문장 전체를 분류하는 모델을 자동으로 불러오는 클래스 
name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

text = "I was not expecting much, but this movie was wonderful."
inputs = tokenizer(text, return_tensors="pt")     # 파이토치 tensor       # 1. text -> token ids
print(inputs)
print()
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    logits = model(**inputs).logits                      # 2. ids -> logits, shape (1, 2)
probs = torch.softmax(logits, dim=-1)[0]                 # 3. logits -> probabilities
pred = int(probs.argmax())
print(logits, probs)
print(model.config.id2label[pred], f"{probs[pred]:.3f}")  # same label and score as the pipeline

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11496.70it/s]

{'input_ids': tensor([[ 101, 1045, 2001, 2025, 8074, 2172, 1010, 2021, 2023, 3185, 2001, 6919,
         1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

['[CLS]', 'i', 'was', 'not', 'expecting', 'much', ',', 'but', 'this', 'movie', 'was', 'wonderful', '.', '[SEP]']
tensor([[-4.3397,  4.6646]]) tensor([1.2287e-04, 9.9988e-01])
POSITIVE 1.000


## 4. The Hub from Python

Every model is a git repository on the Hub. `huggingface_hub` lists its files without downloading them.

In [ ]:
from huggingface_hub import list_repo_files
for f in list_repo_files("distilbert/distilbert-base-uncased-finetuned-sst-2-english"):
    print(f)

#허깅 페이스 hub에 있는 모델 저장소의 파일 목록만 확인 하는 코드 
#list_repo_files(...)
#-> 저장소에 어떤 파일이 있는지 확인

'[Errno 8] nodename nor servname provided, or not known' thrown while requesting GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main
Retrying in 1s [Retry 1/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main
Retrying in 2s [Retry 2/5].


.gitattributes
README.md
config.json
map.jpeg
model.safetensors
onnx/added_tokens.json
onnx/config.json
onnx/model.onnx
onnx/special_tokens_map.json
onnx/tokenizer.json
onnx/tokenizer_config.json
onnx/vocab.txt
pytorch_model.bin
rust_model.ot
tf_model.h5
tokenizer_config.json
vocab.txt


`config.json` holds the architecture, `model.safetensors` the weights, and `tokenizer.json` the tokenizer.
`README.md` is the model card: training data, intended use, limitations, and license.

## Try it

1. Run the sentiment pipeline on five sentences of your own, including one with sarcasm. Which ones fail?
2. Change the zero-shot `candidate_labels` to labels that overlap (e.g., `"phones"` and `"technology"`). How do the scores change?
3. In Section 3, print `tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])`. Which tokens did the tokenizer add?

In [14]:
from transformers import pipeline

clf= pipeline("sentiment-analysis", model= "distilbert/distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "I absolutely loved this movie.",
    "The acting was terrible and boring.",
    "It was okay, nothing special.",
    "What a fantastic movie... I almost fell asleep.",
    "This is the best phone I have ever used."
]

for s ,out in zip(sentences,clf(sentences)):
    print(out["label"],out["score"],"|",s )

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 13501.12it/s]


POSITIVE 0.9998776912689209 | I absolutely loved this movie.
NEGATIVE 0.9997758269309998 | The acting was terrible and boring.
NEGATIVE 0.9820851683616638 | It was okay, nothing special.
NEGATIVE 0.9189872741699219 | What a fantastic movie... I almost fell asleep.
POSITIVE 0.999725878238678 | This is the best phone I have ever used.


In [15]:
from transformers import pipeline

zs= pipeline("zero-shot-classification", model= "facebook/bart-large-mnli")

zs(
    "The new phone has a brighter screen and a battery that lasts two days.",
    candidate_labels=["phones", "technology", "sports"]
)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7365.28it/s]


{'sequence': 'The new phone has a brighter screen and a battery that lasts two days.',
 'labels': ['technology', 'phones', 'sports'],
 'scores': [0.6322649717330933, 0.35912275314331055, 0.008612241595983505]}